In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import datetime
import requests_cache
from tqdm import tqdm


In [2]:
cache_name = 'yfinance_cache'
expire_after = datetime.timedelta(days=1) # Cache expires after 1 day

# Create a cached session
session = requests_cache.CachedSession(
    cache_name=cache_name,
    backend='sqlite',
    expire_after=expire_after
)

In [3]:
class Dataloader:
    def __init__(self, period, session, extra_obs, max_episode_length=1000):
        self.session = session
        self.period = period
        self.extra_obs = extra_obs
        self.max_episode_length = max_episode_length
        self.vix_data = yf.Ticker("^VIX", session=self.session).history(period=period, auto_adjust=True)
        self.gspc_data = yf.Ticker("^GSPC", session=self.session).history(period=period, auto_adjust=True)
        self.random_symbols = [
            "NVDA", "AMZN", "GOOGL", "MSFT", "AAPL", "META", "ADBE",
            "NFLX", "TSLA", "JPM", "V", "UNH"
        ]
        
        self.random_symbols_extend = [
            # Communication Services
            "T", "VZ", "META", "CMCSA",
            # Consumer Discretionary
            "AMZN", "TSLA", "NKE", "MCD", "HD",
            # Consumer Staples
            "PG", "KO", "PEP", "WMT",
            # Energy
            "XOM", "CVX", "COP", "SLB", "OXY",
            # Financials
            "JPM", "BAC", "WFC", "GS", "C",
            # Healthcare
            "JNJ", "PFE", "MRK", "UNH", "ABT",
            # Industrials
            "BA", "CAT", "HON", "MMM", "UNP",
            # Information Technology
            "AAPL", "MSFT", "GOOGL", "NVDA", "ADBE",
            # Materials
            "SHW", "DD", "LIN", "NEM",
            # Real Estate
            "AMT", "PLD", "SPG", "AVB",
            # Utilities
            "NEE", "DUK", "SO", "D"
        ]
    
    def dataloader(self): 
        all_stock_data = {}
        for symbol in tqdm(self.random_symbols_extend, desc="Fetching stock data"):
                all_stock_data[symbol] = self.batch_fetch_data(symbol)
        return all_stock_data
    
    def batch_fetch_data(self, stock):
        # print(f"[INFO] Fetching data for {stock}...")
        try:
            stock_ticker = yf.Ticker(stock, session=self.session)
            stock_data = stock_ticker.history(period=self.period, auto_adjust=True)
            if self.extra_obs:
                stock_df = stock_data[['Open', 'High', 'Low', 'Close', 'Volume']].copy() if not stock_data.empty else pd.DataFrame(index=stock_data.index)
                vix_df = self.vix_data[['Close']].rename(columns={'Close': 'VIX_Close'}).copy() if not self.vix_data.empty else pd.DataFrame(index=self.vix_data.index)
                gspc_df = self.gspc_data[['Close']].rename(columns={'Close': 'GSPC_Close'}).copy() if not self.gspc_data.empty else pd.DataFrame(index=self.gspc_data.index)
                merged_data = pd.concat([stock_df, vix_df, gspc_df], axis=1, join='outer')
                if 'Close' not in merged_data.columns and not stock_df.empty:
                    print("[ERROR] Primary stock 'Close' column missing after outer join.")
            else:
                stock_df = stock_data[['Open', 'High', 'Low', 'Close', 'Volume']].copy() if not stock_data.empty else pd.DataFrame(index=stock_data.index)
                merged_data = stock_df
            merged_data = merged_data.groupby(merged_data.index.date).first()
            merged_data.index = pd.to_datetime(merged_data.index) # Ensure index is datetime
            
            merged_data = merged_data.ffill(limit=3)
            # Handle NaNs in critical columns
            if 'Volume' in merged_data.columns:
                merged_data['Volume'].fillna(0.0, inplace=True) # Fill NaN volume with 0
            merged_data.dropna(subset=['Close'], inplace=True) # Drop rows ONLY if 'Close' is missing
            min_required_length = 35 + self.max_episode_length
            if len(merged_data) < min_required_length:
                print(f"[WARNING] Insufficient merged data after processing for {stock} (Final Length: {len(merged_data)}, Required: {min_required_length}).")
                return None
            
            return merged_data
        except Exception as e:
            print(f"[ERROR] Exception during data fetch/process for {stock}: {e}")
            return None

dataloader = Dataloader(period="max", session=session, extra_obs=True)
dataloader = dataloader.dataloader()

Fetching stock data:   0%|          | 0/50 [00:00<?, ?it/s]/var/folders/qw/5rcwl4d96zvcq7zxv7fzt3m80000gn/T/ipykernel_98470/3866314840.py:66: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_data['Volume'].fillna(0.0, inplace=True) # Fill NaN volume with 0
Fetching stock data:   2%|▏         | 1/50 [00:00<00:18,  2.60it/s]/var/folders/qw/5rcwl4d96zvcq7zxv7fzt3m80000gn/T/ipykernel_98470/3866314840.py:66: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace

In [4]:
#put the stocks inot buckets of 10 each based on volatility ranked
def rank_stocks_by_volatility(stock_data):
    volatility_dict = {}
    for stock, data in stock_data.items():
        if data is not None:
            daily_returns = data['Close'].pct_change()
            volatility = np.std(daily_returns)
            volatility_dict[stock] = volatility

    # Sort stocks by volatility
    sorted_stocks = sorted(volatility_dict.items(), key=lambda x: x[1], reverse=True)
    
    # Create buckets of 10 stocks each
    buckets = [sorted_stocks[i:i + 10] for i in range(0, len(sorted_stocks), 10)]
    
    return buckets
buckets = rank_stocks_by_volatility(dataloader)
# Print the buckets
for i, bucket in enumerate(buckets):
    print(f"Bucket {i + 1}:")
    for stock, volatility in bucket:
        print(f"  {stock}: {volatility:.4f}")
    print()
    

Bucket 1:
  NVDA: 0.0379
  TSLA: 0.0366
  AMZN: 0.0352
  ADBE: 0.0304
  AMT: 0.0286
  AAPL: 0.0278
  META: 0.0252
  C: 0.0251
  UNH: 0.0250
  NEM: 0.0250

Bucket 2:
  BAC: 0.0237
  GS: 0.0229
  PLD: 0.0229
  HD: 0.0228
  SLB: 0.0228
  OXY: 0.0225
  JPM: 0.0222
  CMCSA: 0.0222
  NKE: 0.0216
  SPG: 0.0215

Bucket 3:
  BA: 0.0215
  MSFT: 0.0210
  WFC: 0.0208
  COP: 0.0207
  DD: 0.0196
  GOOGL: 0.0193
  CAT: 0.0186
  WMT: 0.0184
  MCD: 0.0183
  SHW: 0.0183

Bucket 4:
  HON: 0.0182
  AVB: 0.0177
  LIN: 0.0175
  UNP: 0.0174
  PFE: 0.0173
  CVX: 0.0162
  ABT: 0.0160
  MRK: 0.0156
  T: 0.0156
  PEP: 0.0153

Bucket 5:
  VZ: 0.0149
  MMM: 0.0148
  XOM: 0.0145
  KO: 0.0144
  JNJ: 0.0144
  NEE: 0.0137
  D: 0.0135
  DUK: 0.0135
  PG: 0.0134
  SO: 0.0127

